In [1]:
import sionna.rt

print(dir(sionna.rt.scene))

['AntennaArray', 'Boltzmann', 'Camera', 'DEFAULT_BANDWIDTH', 'DEFAULT_FREQUENCY', 'DEFAULT_PREVIEW_BACKGROUND_COLOR', 'DEFAULT_TEMPERATURE', 'List', 'Previewer', 'RadioMaterialBase', 'Receiver', 'Scene', 'Transmitter', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'annotations', 'box', 'box_knife', 'box_one_screen', 'box_two_screens', 'contextlib', 'double_reflector', 'dr', 'edit_scene_shapes', 'etoile', 'files', 'floor_wall', 'florence', 'load_scene', 'load_scene_from_string', 'low_poly_car', 'matplotlib', 'mi', 'munich', 'np', 'os', 'plt', 'process_xml', 'radio_map_color_mapping', 'render', 'san_francisco', 'scenes', 'simple_reflector', 'simple_street_canyon', 'simple_street_canyon_with_cars', 'simple_wedge', 'sionna', 'speed_of_light', 'sphere', 'triple_reflector']


In [5]:
import mitsuba as mi
print(mi.variant())

print(sionna.__version__)

llvm_ad_mono_polarized
1.2.2


In [6]:
import inspect
from sionna.rt import Scene

print(inspect.signature(Scene))

help(Scene)

(mi_scene: 'mi.Scene | None' = None, remove_duplicate_vertices: 'bool' = False)
Help on class Scene in module sionna.rt.scene:

class Scene(builtins.object)
 |  Scene(mi_scene: 'mi.Scene | None' = None, remove_duplicate_vertices: 'bool' = False)
 |  
 |  A scene contains everything that is needed for radio propagation simulation
 |  and rendering.
 |  
 |  A scene is a collection of multiple instances of
 |  :class:`~sionna.rt.SceneObject` which define the geometry and materials of
 |  the objects in the scene. It also includes transmitters
 |  (:class:`~sionna.rt.Transmitter`) and
 |  receivers (:class:`~sionna.rt.Receiver`).
 |  
 |  A scene is instantiated by calling :func:`~sionna.rt.load_scene()`.
 |  
 |  `Example scenes <https://nvlabs.github.io/sionna/rt/api/scene.html#examples>`_ can be loaded as follows:
 |  
 |  .. code-block:: python
 |  
 |      from sionna.rt import load_scene
 |      scene = load_scene(sionna.rt.scene.munich)
 |      scene.preview()
 |  
 |  .. figure:: 

In [1]:
import mitsuba as mi

mi.set_variant("llvm_ad_mono_polarized")

import sionna.rt
from sionna.rt import *


In [2]:
for k,v in scene.objects.items():
    print(k, type(v))
    break

Heilig_Geist-itu_marble <class 'sionna.rt.scene_object.SceneObject'>


In [57]:
scene = load_scene(sionna.rt.scene.box)
scene.preview(
    clip_at=0,
    clip_plane_orientation=(0,1,0)
)

In [58]:
scene.tx_array = PlanarArray(
    num_rows=1,
    num_cols=1,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="iso",
    polarization="V"
)

scene.rx_array = PlanarArray(
    num_rows=1,
    num_cols=1,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="iso",
    polarization="V"
)

tx = Transmitter(
    name="tx",
    position=[-4, -4, 1]
)

rx0 = Receiver(
    name="rx0",
    position=[4, 4, 1.0]
)

rx1 = Receiver(
    name="rx1",
    position=[-4, 4, 1.0]
)

rx2 = Receiver(
    name="rx2",
    position=[4, -4, 1.0]
)

scene.add(tx)
scene.add(rx0)
scene.add(rx1)
scene.add(rx2)

print(scene.transmitters.keys())
print(scene.receivers.keys())

dict_keys(['tx'])
dict_keys(['rx0', 'rx1', 'rx2'])


In [59]:
from sionna.rt import PathSolver

# 1. 전파 경로(Ray Tracing) 계산 - 최신 API 적용
# PathSolver 인스턴스를 생성한 뒤 맵(scene) 정보를 넘겨줌
solver = PathSolver()

paths = solver(
    scene=scene,
    max_depth=2 # 이게 반사강도임 얼마나 많이 튕길거냐 
)

print(type(paths))
print(paths.a)
len(paths.a)
scene.preview(paths=paths)

<class 'sionna.rt.path_solvers.paths.Paths'>
([[[[[-0.000451164, -0.000451164, -0.000451164, .. 17 skipped .., -0.000525127, -0.000525127, 0.000451145]]]],
 [[[[-0.000531953, 0.000668229, -0.000531954, .. 17 skipped .., 0.000531987, 0, 0]]]],
 [[[[0.000852026, 0.000602305, -0.000681486, .. 17 skipped .., 0.000531987, 0, 0]]]]], [[[[[2.50008e-07, 2.50009e-07, 2.50009e-07, .. 17 skipped .., 7.5349e-07, 7.5349e-07, -2.68859e-07]]]],
 [[[[3.02473e-07, -1.55218e-07, 3.02475e-07, .. 17 skipped .., -2.68883e-07, 0, 0]]]],
 [[[[0, -1.6809e-07, 1.34495e-07, .. 17 skipped .., -2.68884e-07, 0, 0]]]]])


In [8]:
import numpy as np

# 1. 튜플 언패킹: 진폭(a)과 지연시간(tau) 분리
a, tau = paths.cir()

# 2. NumPy 배열로 변환 (Sionna 텐서를 변환)
a_np = np.array(a)
tau_np = np.array(tau)

# 3. 데이터 형태(Shape) 확인 
# 보통 [batch size, num_rx, num_rx_ant, num_tx, num_tx_ant, num_paths] 형태를 가짐
print("Amplitude shape:", a_np.shape)
print("Delay shape:", tau_np.shape)

# 4. 파이토치 학습을 위해 파일로 저장
np.save('csi_amplitude.npy', a_np)
np.save('csi_delay.npy', tau_np)

print("✅ 학습용 데이터 추출 및 저장 완료!")

Amplitude shape: (2, 3, 1, 1, 1, 21, 1)
Delay shape: (3, 1, 21)
✅ 학습용 데이터 추출 및 저장 완료!


In [13]:
for rx in range(3):

    gains = np.abs(
        a_np[0, rx, 0, 0, 0, :, 0]
    )

    print(f"RX {rx}")
    print("Strongest:", gains.max())
    print("Num Paths:", np.sum(gains > 1e-6))

RX 0
Strongest: 0.0034081032
Num Paths: 21
RX 1
Strongest: 0.002272069
Num Paths: 21
RX 2
Strongest: 0.0017040516
Num Paths: 21


In [14]:
a_np = np.array(a)
tau_np = np.array(tau)

for rx in range(3):

    gains = np.abs(
        a_np[0,rx,0,0,0,:,0]
    )

    delays = tau_np[rx,0,:]

    print(f"\nRX {rx}")
    print("Strongest:", gains.max())
    print("Delay min:", delays.min()*1e9, "ns")
    print("Delay max:", delays.max()*1e9, "ns")


RX 0
Strongest: 0.0006024732
Delay min: 0.0 ns
Delay max: 59.39686786859966 ns

RX 1
Strongest: 0.0008520258
Delay min: -1000000000.0 ns
Delay max: 66.71281482795166 ns

RX 2
Strongest: 0.0008520258
Delay min: -1000000000.0 ns
Delay max: 66.71281482795166 ns


In [36]:
import sionna.rt as rt

print([x for x in dir(rt.scene) if not x.startswith("_")])

['AntennaArray', 'Boltzmann', 'Camera', 'DEFAULT_BANDWIDTH', 'DEFAULT_FREQUENCY', 'DEFAULT_PREVIEW_BACKGROUND_COLOR', 'DEFAULT_TEMPERATURE', 'List', 'Previewer', 'RadioMaterialBase', 'Receiver', 'Scene', 'Transmitter', 'annotations', 'box', 'box_knife', 'box_one_screen', 'box_two_screens', 'contextlib', 'double_reflector', 'dr', 'edit_scene_shapes', 'etoile', 'files', 'floor_wall', 'florence', 'load_scene', 'load_scene_from_string', 'low_poly_car', 'matplotlib', 'mi', 'munich', 'np', 'os', 'plt', 'process_xml', 'radio_map_color_mapping', 'render', 'san_francisco', 'scenes', 'simple_reflector', 'simple_street_canyon', 'simple_street_canyon_with_cars', 'simple_wedge', 'sionna', 'speed_of_light', 'sphere', 'triple_reflector']


In [60]:
from sionna.rt import SceneObject, ITURadioMaterial
import sionna.rt as rt

human_mat = ITURadioMaterial(
    name="human_mat",
    itu_type="concrete",   # 일단 아무거나
    thickness=0.1
)

human = SceneObject(
    fname=rt.scene.sphere,
    name="human_proxy",
    radio_material=human_mat
)

print(human)

In [61]:
new_mi_scene = rt.scene.edit_scene_shapes(
    scene,
    add=human
)

In [62]:
solver = PathSolver()

# 원본
paths0 = solver(
    scene=scene,
    max_depth=2
)

a0, tau0 = paths0.cir()

# sphere 추가
with scene.use_mi_scene(new_mi_scene):

    paths1 = solver(
        scene=scene,
        max_depth=2
    )

    a1, tau1 = paths1.cir()

print("base :", np.count_nonzero(np.abs(a0) > 0))
print("sphere:", np.count_nonzero(np.abs(a1) > 0))

print("base amp :", np.mean(np.abs(a0)))
print("sphere amp:", np.mean(np.abs(a1)))

base : 127
sphere: 117
base amp : 0.00032203784
sphere amp: 0.00032889578


In [63]:
with scene.use_mi_scene(new_mi_scene):
    scene.preview(paths=paths1)

In [65]:
print(human.mi_mesh.bbox())
print(human.position)

BoundingBox3f[
  min = [-0.987231, -0.987231, -0.997577],
  max = [0.987231, 0.987231, 0.997577]
]
[[2.38419e-07, 1.49012e-07, 0]]


In [ ]:
def build_scene(human_pos=None):

    scene = load_scene()

    if human_pos is not None:

        x,y,z = human_pos

        human = create_human(
            x,
            y,
            z
        )

        new_scene = rt.scene.edit_scene_shapes(
            scene,
            add=human
        )

        return scene, new_scene

    return scene, None

In [80]:
from sionna.rt import RadioMaterial

human_mat = RadioMaterial(
    name="human_mat",
    relative_permittivity=38.0,
    conductivity=1.5
)

import mitsuba as mi
import drjit as dr

from sionna.rt import SceneObject
import sionna.rt as rt

human = SceneObject(
    fname=rt.scene.sphere,
    name="human_proxy",
    radio_material=human_mat
)

mesh = human.mi_mesh
SHIFT_X = 0.0
SHIFT_Y = 0.0
SHIFT_Z = 1.0

v = mesh.vertex_positions_buffer()

vertices = dr.unravel(mi.Point3f, v)

vertices += mi.Vector3f(
    SHIFT_X,
    SHIFT_Y,
    SHIFT_Z
)

mesh_params = mi.traverse(mesh)

mesh_params["vertex_positions"] = dr.ravel(vertices)
mesh_params.update()

human_shifted = SceneObject(
    mi_mesh=mesh,
    name="human_shifted",
    radio_material=human_mat
)

new_scene_shifted = rt.scene.edit_scene_shapes(
    scene,
    add=human_shifted
)

with scene.use_mi_scene(new_scene_shifted):

    print("===== SHAPES =====")

    for s in scene.mi_scene.shapes():
        print(s.id())

ValueError: Cannot add object of type (<class 'mitsuba.Mesh'>) with ID "human_shifted" because this ID is already used in the scene.

In [ ]:
with scene.use_mi_scene(new_scene_shifted):

    scene.render_to_file(
        camera="top_cam",
        filename="human_test.png"
    )